In [36]:
import os
import pyreadr
import json
import csv
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from notebook_utils import NOTEBOOK_DIR
from cityscaper.utils import read_rds_to_df, resolve_path, sorted_columns, latlon_filter, geojson_rds_to_json, geojson_to_parcel_bounds
from cityscaper.constants import EXPORT_FIELDS
from cityscaper.geom import kml_from_latlon

DATA_DIR = NOTEBOOK_DIR.parent / "data"
OUTPUT_DIR = DATA_DIR / "output"

%matplotlib inline

In [27]:
# Load reference data for exploration
raw_geom_geojson = geojson_rds_to_json(DATA_DIR / "sf_map_unfiltered.RDS")
clean_geom = geojson_to_parcel_bounds(raw_geom_geojson)
geom_series = pd.Series(clean_geom)

fname = DATA_DIR / "five_rezonings_nongeo.RDS"
rezoning_base = read_rds_to_df(DATA_DIR / "five_rezonings_nongeo_unfiltered.RDS", index_cols='mapblklot')
print('\n'.join(sorted_columns(rezoning_base)))
rezoning_base.loc['3561025',['lat', 'lng', 'year', 'pdev_baseline_1yr', 'ex_height2024']]

ACRES
ADDRESS
affh2023
college_dist
commercial_dist
Const_FedReserve_Real
Developed
econ_affh
Envelope_1000
EX_GP_TYPE
ex_height2024
EX_USE
EX_ZONING
expected_units_baseline_if_dev
expected_units_skyscraper_if_dev
Historic
is_corner
lat
lng
M1_CAP
M1_GP_TYPE
M1_MAXDENS
M1_ZONING
M2_CAP
M2_GP_TYPE
M2_MAXDENS
M2_ZONING
M3_CAP
M3_GP_TYPE
M3_MAXDENS
M3_ZONING
M4_height
M4_ZONING
M5_height
M5_ZONING
MapBlkLot_Master
nhood
park_dist
pdev_baseline_1yr
pdev_skyscraper_1yr
peg
Residential_Dummy
sb330_applies
sup_dist_name
sup_name
transit_dist
transit_dist_bart
transit_dist_caltrain
transit_dist_rapid
transit_dist_rapid_stops
Upzone_Ratio
VACANT
year
Zillow_Price_Real
ZONING
zp_DensRestMulti
zp_FormBasedMulti
zp_OfficeComm
zp_PDRInd
zp_Public
zp_Redev
zp_RH2
zp_RH3_RM1


lat                   37.764429
lng                 -122.434481
year                     2016.0
pdev_baseline_1yr      0.000168
ex_height2024              40.0
Name: 3561025, dtype: object

In [38]:
input_csv = '/Users/eric/Desktop/rezoning_frontier.csv'
frontier_data = pd.read_csv('/Users/eric/Desktop/rezoning_frontier.csv', dtype={'mapblklot':str}, index_col='mapblklot')
frontier_data['development_study_year'] = 1
frontier_data = frontier_data[['developed_height', 'lot_coverage_discount', 'height', 'development_study_year']]
frontier_data['lot_coverage_discount'] = 0.85
frontier_data = frontier_data.join(rezoning_base, how='left')
frontier_data['ground_floor'] = frontier_data['ACRES'] * 43560 * frontier_data['lot_coverage_discount']

frontier_data_export_fname = '/Users/eric/Desktop/rezoning_frontier.csv'
frontier_data[EXPORT_FIELDS + ['development_study_year']].to_csv(frontier_data_export_fname)

with open(frontier_data_export_fname, newline="") as f:
    parcel_specs = list(csv.DictReader(f))

kml = kml_from_latlon(parcel_specs=parcel_specs, geom_data=clean_geom)
kml_fname = '/Users/eric/Desktop/rezoning_frontier.kml'
with open(kml_fname, 'w') as kml_file:
    kml_file.write(kml)

In [29]:
clean_geom

{'0001001': [[[-122.422017, 37.808485],
   [-122.422089, 37.80884],
   [-122.421115, 37.808808],
   [-122.421076, 37.808606],
   [-122.422017, 37.808485]]],
 '0002001': [[[-122.420839, 37.808636],
   [-122.420871, 37.808801],
   [-122.419825, 37.808767],
   [-122.420839, 37.808636]]],
 '0014001': [[[-122.413889, 37.807597],
   [-122.414041, 37.808344],
   [-122.412642, 37.808519],
   [-122.41248, 37.807777],
   [-122.413889, 37.807597]]],
 '0015001': [[[-122.412245, 37.807803],
   [-122.412394, 37.808483],
   [-122.410896, 37.80826],
   [-122.410835, 37.807981],
   [-122.412245, 37.807803]]],
 '0016001': [[[-122.410606, 37.808004],
   [-122.410654, 37.808226],
   [-122.409845, 37.808101],
   [-122.410606, 37.808004]]],
 '0017002': [[[-122.408854, 37.807271],
   [-122.408984, 37.807927],
   [-122.407756, 37.807409],
   [-122.408854, 37.807271]]],
 '0018001': [[[-122.410093, 37.807878],
   [-122.409153, 37.807999],
   [-122.409079, 37.807625],
   [-122.410019, 37.807507],
   [-122.410093

In [26]:
frontier_data[EXPORT_FIELDS].

,developed_height,lot_coverage_discount,ground_floor,ACRES,height,Historic,Residential_Dummy,ZONING
mapblklot,,,,,,,,
0342015,85.0,0.85,4708.728564,0.127174,85.0,1,0,NaN
3703003,95.0,0.85,1912.504150,0.051653,120.0,1,0,NaN
3703056,160.0,0.85,5375.844922,0.145191,85.0,1,0,NaN
3703078,85.0,0.85,1806.250830,0.048783,85.0,1,0,NaN
3704051,60.0,0.85,1498.373193,0.040468,85.0,1,1,NaN
3704053,55.0,0.85,1729.314209,0.046705,65.0,1,1,NaN


In [14]:
rezoning_base.loc[ref_lots, EXPORT_FIELDS]

KeyError: "['developed_height', 'lot_coverage_discount', 'ground_floor', 'height'] not in index"

In [ ]:
geom_select = [-122.43270,37.76874,-122.43060,37.77047]
rezoning_scenario = "F"  # Family rezoning

rezoning_fname = f"rezoning_{rezoning_scenario}_output.RDS"
rezoning_scenario_data = read_rds_to_df(DATA_DIR / rezoning_fname, index_cols='mapblklot')
assert 'height' in rezoning_scenario_data.columns
assert 'pdev' in rezoning_scenario_data.columns

lots_in_region = latlon_filter(rezoning_scenario_data, *geom_select)
development_candidates = lots_in_region[lots_in_region['ZONING'].notnull()].copy()

modeling_variables = [c for c in development_candidates.columns if 'pdev' in c] + \
                     ['M4_ZONING', 'M5_ZONING', 'M6_ZONING', 'height', 'ZONING','M6_height',
                      'height', 'lot_coverage_discount', 'ground_floor', 'ACRES', 'ex_height2024']
print("\n".join(modeling_variables))
# rezoning_scenario_data.loc['3561025',['M4_ZONING', 'M5_ZONING', 'M6_ZONING', 'height', 'ZONING','M6_height']]

development_candidates[modeling_variables]

In [ ]:
import os
import csv
import json

header = """<?xml version="1.0" encoding="UTF-8"?>
<kml xmlns="http://www.opengis.net/kml/2.2" 
     xmlns:gx="http://www.google.com/kml/ext/2.2">
  <Document>
"""

template = """    <Placemark>
      <name>{mapblklot}</name>
      <description>{height}</description>
      <Polygon>
        <extrude>1</extrude>
        <altitudeMode>relativeToGround</altitudeMode>
        <outerBoundaryIs>
          <LinearRing>
            <coordinates>
{coordinate_string}
            </coordinates>
          </LinearRing>
        </outerBoundaryIs>
      </Polygon>
      <Style>
        <PolyStyle>
          <color>cc0000ff</color> <!-- Red with 80% opacity -->
        </PolyStyle>
      </Style>
    </Placemark>"""

footer="""  </Document>
</kml>"""

geometry_file = os.path.expanduser("~/src/cityscaper/data/sf_map_unfiltered.json")
csv_path = os.path.expanduser("~/Desktop/rezoning_output.csv")
csv_path = os.path.expanduser("~/Desktop/rezoning_geary.csv")
with open(geometry_file, "r") as f:
    geom_data = json.load(f)

with open(csv_path, newline="") as f:
    parcel_specs = list(csv.DictReader(f))

buildings = []
for i, row in enumerate(parcel_specs):
    lot = row["mapblklot"]
    try:
        height = float(row["height"])
        parcel_bounds = geom_data[lot]
        for j, polygon in enumerate(parcel_bounds):
            building_name = f"{lot}_{j+1}"
            vertex_strings = []
            for lat, lng in polygon:
                vertex_strings.append(f"              {lat},{lng},{height*0.3048}")
            coordinate_string="\n".join(vertex_strings)
            building_string = template.format(mapblklot=lot,
                                              coordinate_string=coordinate_string,
                                              height=height
                                             )
            buildings.append(building_string)
    except KeyError as e:
        continue

kml = header + "\n".join(buildings) + footer

print(kml)